# Assignment 1 – Feedforward Neural Network on MNIST

## Task 1 – Load the data
Same parameters and transformations as in the blog post (batch size 64 / 1000, normalisation with mean 0.1307 and std 0.3081).

In [3]:
import time
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

n_epochs = 50
batch_size_train = 64
batch_size_test = 1000
learning_rate = 0.01
momentum = 0.5
log_interval = 10
mean = 0.1307
standardDeviation = 0.3801

random_seed = 1
torch.backends.cudnn.enabled = False
torch.manual_seed(random_seed)

train_loader = torch.utils.data.DataLoader(
    torchvision.datasets.MNIST(
        "./data/",
        train=True,
        download=True,
        transform=torchvision.transforms.Compose(
            [
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Normalize((mean,), (standardDeviation,)),
            ]
        ),
    ),
    batch_size=batch_size_train,
    shuffle=True,
)

test_loader = torch.utils.data.DataLoader(
    torchvision.datasets.MNIST(
        "./data/",
        train=False,
        download=True,
        transform=torchvision.transforms.Compose(
            [
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Normalize((mean,), (standardDeviation,)),
            ]
        ),
    ),
    batch_size=batch_size_test,
    shuffle=True,
)

examples = enumerate(test_loader)
batch_idx, (example_data, example_targets) = next(examples)

example_data.shape

torch.Size([1000, 1, 28, 28])

## Task 2 – Feedforward neural network
* Each image has shape `1×28×28`, so the input layer size is `D_in = 28*28 = 784` and the tensor is flattened in `forward`.
* ReLU in the hidden layer, no convolutions.
* `log_softmax` as output activation (pairs with the NLL loss).

In [4]:
class TwoLayerNet(nn.Module):
    def __init__(self, D_in, H, D_out):
        super(TwoLayerNet, self).__init__()
        self.linear1 = nn.Linear(D_in, H)
        self.linear2 = nn.Linear(H, D_out)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten [N, 1, 28, 28] -> [N, 784]
        h_relu = F.relu(self.linear1(x))
        y_pred = self.linear2(h_relu)
        return F.log_softmax(y_pred, dim=-1)


# D_in is input dimension; H is hidden dimension; D_out is output dimension.
D_in, H, D_out = 28 * 28, 100, 10

model = TwoLayerNet(D_in, H, D_out)
print(model)

TwoLayerNet(
  (linear1): Linear(in_features=784, out_features=100, bias=True)
  (linear2): Linear(in_features=100, out_features=10, bias=True)
)


## Task 3 – Train the network for 50 epochs
Negative log likelihood loss and SGD with the blog post's learning rate and momentum.
`train` and `test` take a `device` argument so the same code can run on CPU and GPU (Task 5).

In [ ]:
def train(model, optimizer, device, epoch, train_losses, train_counter):
    #  
    model.train()

    #
    epoch_loss = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(data)
        if batch_idx % log_interval == 0:
            train_losses.append(loss.item())
            train_counter.append(
                (batch_idx * 64) + ((epoch - 1) * len(train_loader.dataset))
            )
    return epoch_loss / len(train_loader.dataset)


def test(model, device):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction="sum").item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
    test_loss /= len(test_loader.dataset)
    accuracy = 100.0 * correct / len(test_loader.dataset)
    print(
        "Test set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.2f}%)".format(
            test_loss, correct, len(test_loader.dataset), accuracy
        )
    )
    return test_loss, accuracy


def run_training(device):
    torch.manual_seed(random_seed)  # same initial weights and shuffling for every run
    model = TwoLayerNet(D_in, H, D_out).to(device)
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum)
    train_losses, train_counter, epoch_losses = [], [], []

    start = time.perf_counter()
    for epoch in range(1, n_epochs + 1):
        epoch_losses.append(
            train(model, optimizer, device, epoch, train_losses, train_counter)
        )
        if epoch == 1 or epoch % 5 == 0:
            print(
                "Epoch {:2d}/{}  avg. train loss: {:.4f}".format(
                    epoch, n_epochs, epoch_losses[-1]
                )
            )
    if device.type == "cuda":
        torch.cuda.synchronize()  # wait for all GPU work before stopping the clock
    duration = time.perf_counter() - start
    print("Training time on {}: {:.1f} s".format(device, duration))
    return model, train_losses, train_counter, epoch_losses, duration

In [ ]:
cpu = torch.device('cpu')
model_cpu, train_losses, train_counter, epoch_losses, time_cpu = run_training(cpu)

In [ ]:
fig = plt.figure(figsize=(9, 5))
plt.plot(train_counter, train_losses, color='blue', alpha=0.4, linewidth=0.8, label='Train Loss (every 10th batch)')
plt.plot([e * len(train_loader.dataset) for e in range(1, n_epochs + 1)], epoch_losses,
         color='darkblue', marker='o', markersize=3, label='Train Loss (epoch average)')
plt.legend(loc='upper right')
plt.xlabel('number of training examples seen')
plt.ylabel('negative log likelihood loss')
plt.title('Training loss – TwoLayerNet on MNIST (50 epochs)')
plt.ylim(0, 1)
plt.show()

## Task 4 – Test the network on the MNIST test data

In [ ]:
test_loss_cpu, accuracy_cpu = test(model_cpu, cpu)

In [ ]:
with torch.no_grad():
  output = model_cpu(example_data)

fig = plt.figure()
for i in range(6):
  plt.subplot(2,3,i+1)
  plt.tight_layout()
  plt.imshow(example_data[i][0], cmap='gray', interpolation='none')
  plt.title("Prediction: {}".format(output.argmax(dim=1)[i].item()))
  plt.xticks([])
  plt.yticks([])
plt.show()

## Task 5 – Training on the GPU
PyTorch runs on the GPU when the **model** and the **data** live on a CUDA device:
1. Select the device: `device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')`
2. Move the model's parameters once: `model.to(device)` (before creating the optimizer)
3. Move every batch inside the training/test loop: `data, target = data.to(device), target.to(device)`

Because GPU calls are asynchronous, `torch.cuda.synchronize()` is called before the timer stops.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if device.type == 'cuda':
  print(torch.cuda.get_device_name(0))

model_gpu, _, _, _, time_gpu = run_training(device)
test_loss_gpu, accuracy_gpu = test(model_gpu, device)

In [ ]:
print('{:<6} {:>12} {:>10} {:>10}'.format('Device', 'Train time', 'Test loss', 'Accuracy'))
print('{:<6} {:>10.1f} s {:>10.4f} {:>9.2f}%'.format('CPU', time_cpu, test_loss_cpu, accuracy_cpu))
print('{:<6} {:>10.1f} s {:>10.4f} {:>9.2f}%'.format('GPU', time_gpu, test_loss_gpu, accuracy_gpu))
print('Speed-up: {:.2f}x'.format(time_cpu / time_gpu))

**Discussion:**
* **Task 4:** After 50 epochs the network reaches **97.78 % accuracy** with an average test loss of **0.0718** on the 10,000 MNIST test images. The training loss (0.0108) is clearly lower than the test loss, i.e. the network starts to overfit slightly – the blog post's CNN with dropout does not have that gap.
* **Task 5:** CPU: 367.8 s, GPU (RTX 5060 Ti): 374.6 s → **no speed-up (0.98x)**. The network has only ~80k parameters and the batches are small (64), so the matrix multiplications are trivial for both devices. Training time is dominated by the `DataLoader` (reading every image, `ToTensor` and `Normalize` on the CPU for each batch) and the per-batch overhead of copying the data to the GPU and launching kernels. A GPU only pays off for larger models/batches (e.g. CNNs) or if the whole dataset is preloaded onto the GPU.
* With the same seed, CPU and GPU produce practically identical losses; the tiny differences in the last digits come from different floating-point implementations.